# Product Analytics/Experimentation Case Study
### Business Problem & Experiment

> An e-commerce platform is testing whether a re-designed checkout call-to-action (CTA) can improve conversion rates.
The existing checkout CTA is a grey "**Proceed to Checkout**" button while the proposed variant is a high contrast-green "**Buy Now - Free Returns**" CTA.

> **The Business Question:** Does the redesign checkout CTA improve conversion without negatively affecting revenue per session?

> **Experiment Design:**

1.   Control (A):
  *   CTA:	Grey - "Proceed to Checkout"
  *   Users: 5000
  *   Test period: Mar 3 - 30, 2026

2.   Variant (B):
  *   CTA: Green - "Buy Now - Free Returns"
  *   Users: 5000
  *   Test period: Mar 3 - 30, 2026

> Users were randomly assigned to either the control or variant experience.

> **Primary Metrics**
*   Conversion Rate (CVR): proportion of sessions resulting in a purchase.
*   Revenue per Session: revenue generated per user session.

> **Secondary Metrics**
*   Bounce Rate
*   Add-to-Cart Rate (ATCR)
*   Pages Viewed
*   Funnel progression
*   Time to Conversion

> This analysis evaluates both statistical significance and practical significance to determine whether the observed differences are meaningful for product decision-making.

### Dataset

The analysis uses a simulated e-commerce session-level dataset containing 10,000 users, evenly split between the control and variant groups.

Each row represents a user session and contains information about the user's experiment assignment, demographics, acquisition channel, engagement, cart activity, conversion, and revenue.












### Setup, Import Libraries & Load Dataset

In [1]:
# Connect to drive
from google.colab import drive
drive.mount("/content/drive/")

Mounted at /content/drive/


NOTE: REVISE ALL INSIGHTS!!!!

In [14]:
# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns
sns.set()
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from scipy import stats

In [3]:
# Load dataset
df = pd.read_excel('/content/drive/MyDrive/A-B Testing: Checkout Button Design/A-B_Test_v3.xlsx', sheet_name='Raw Data')
df.head()

,user_id,variant,session_date,session_hour,device,channel,country,age_group,user_type,pages_viewed,session_duration_sec,items_in_cart,cart_value_usd,bounced,add_to_cart_click,converted,revenue_usd,time_to_convert_sec
0,B102653,B,2025-03-03,20,Desktop,Organic Search,Ethiopia,45-54,Returning,8,263,3,61.64,0,1,1,58.56,72.0
1,B102865,B,2025-03-13,15,Desktop,Organic Search,Ghana,18-24,Returning,7,193,2,37.26,0,1,1,35.86,64.0
2,C103226,Control,2025-03-19,11,Mobile,Paid Social,Nigeria,35-44,New,1,13,1,10.17,1,0,0,0.00,NaN
3,B100912,B,2025-03-21,8,Tablet,Direct,Nigeria,35-44,New,9,257,3,33.42,0,0,0,0.00,NaN
4,B103237,B,2025-03-28,13,Mobile,Organic Search,Ghana,25-34,Returning,1,15,2,43.90,1,0,0,0.00,NaN


### Data Quality Checks

In [ ]:
# Dataset dimensions
print(f"Dataset shape: {df.shape}")

# Check missing values
print(f"\nMissing Values:\n{df.isnull().sum()}")

# Duplicated values
print(f"\nDuplicated records: {df.duplicated().sum()}")

# Experimental allocation
print(f"\nExperiment Allocation:\n{df['variant'].value_counts().sort_index(ascending=False)}")

# Inspecting binary variables
binary_cols = ['bounced', 'add_to_cart_click', 'converted']

print("\nInspecting binary variables:")
for col in binary_cols:
  print(df[col].value_counts().sort_index(), "\n")

# Check whether each user appears once
print("Validating uniqueness of records")
print(f"No. of unique records: {df['user_id'].nunique()}")
print(f"Length of df: {len(df)}\n")

# Checking time_to_convert_sec against converted
result = df.groupby('converted')['time_to_convert_sec'].agg(
    ['count', 'min', 'max']
)
print(result)
print("\nAll good; converted == 0 don't have time_to_convert_sec aggregations")

Dataset shape: (10000, 18)

Missing Values:
user_id                    0
variant                    0
session_date               0
session_hour               0
device                     0
channel                    0
country                    0
age_group                  0
user_type                  0
pages_viewed               0
session_duration_sec       0
items_in_cart              0
cart_value_usd             0
bounced                    0
add_to_cart_click          0
converted                  0
revenue_usd                0
time_to_convert_sec     7785
dtype: int64

Duplicated records: 0

Experiment Allocation:
variant
Control    5000
B          5000
Name: count, dtype: int64

Inspecting binary variables:
bounced
0    7800
1    2200
Name: count, dtype: int64 

add_to_cart_click
0    6426
1    3574
Name: count, dtype: int64 

converted
0    7785
1    2215
Name: count, dtype: int64 

Validating uniqueness of records
No. of unique records: 10000
Length of df: 10000

           coun

## Exploratory Data Analysis (EDA)

> EDA will shed more light on user behavior across both experiment groups; further it will identify patterns and trends that may influence conversion, bounce rate, and revenue optimization.

> This section examines overall permaformance of both variants, and differences across device, marketing channel, user engagement, cart activity, age groups, and country.





### Overall Performance by Variant

In [ ]:
# Overall performance by variant
variant_summary = df.groupby('variant').agg(
    n_users = ('user_id', 'count'),
    conversions = ('converted', 'sum'),
    revenue = ('revenue_usd', 'sum'),
    bounced = ('bounced', 'sum'),
    add_to_cart = ('add_to_cart_click', 'sum'),
    avg_pages_viewed = ('pages_viewed', 'mean')
)

# Calc CVR
variant_summary['conversion_rate'] = (
    variant_summary['conversions'] / variant_summary['n_users']
)

# Calc bounce rate
variant_summary['bounce_rate'] = (
    variant_summary['bounced'] / variant_summary['n_users']
)

# Calc add to cart rate (ATCR)
variant_summary['add_to_cart_rate'] = (
    variant_summary['add_to_cart'] / variant_summary['n_users']
)

# Calc revenue per session
variant_summary['revenue_per_session'] = (
    variant_summary['revenue'] / variant_summary['n_users']
)

variant_summary.round(3)

,n_users,conversions,revenue,bounced,add_to_cart,avg_pages_viewed,conversion_rate,bounce_rate,add_to_cart_rate,revenue_per_session
variant,,,,,,,,,,
B,5000,1227,58758.16,1080,1884,3.805,0.245,0.216,0.377,11.752
Control,5000,988,48575.36,1120,1690,3.777,0.198,0.224,0.338,9.715


### Conversion and Bounce Rates by Device

> Does the new checkout design behave differently across devices?



In [28]:
device_summary = df.groupby(['device', 'variant']).agg(
    users = ('user_id', 'count'),
    conversions = ('converted', 'sum'),
    bounced = ('bounced', 'sum')
).reset_index()

# Calc conversion rate (CVR)
device_summary['conversion_rate'] = (
    device_summary['conversions'] / device_summary['users']
)*100

# Calc bounce rate
device_summary['bounce_rate'] = (
    device_summary['bounced'] / device_summary['users']
)*100

# Visualize CVR
fig = px.bar(
    device_summary,
    x = 'device',
    y = 'conversion_rate',
    color = 'variant',
    barmode = 'group',
    text='conversion_rate',
    title='<b>Conversion Rate (CVR) by Device<b>'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')

fig.update_layout(
    yaxis_title='Conversion Rate (%)',
    xaxis_title='Device',
    height=500
)

fig.show()

# Visualize bounce rate
fig = px.bar(
    device_summary,
    x = 'device',
    y = 'bounce_rate',
    color = 'variant',
    barmode = 'group',
    text='bounce_rate',
    title='<b>Bounce Rate by Device<b>'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')

fig.update_layout(
    yaxis_title='Bounce Rate (%)',
    xaxis_title='Device',
    height=500
)

fig.show()

### Conversion Rate and Bounce Rate by Marketing Channel

In [29]:
channel_summary = df.groupby(['channel', 'variant']).agg(
    users = ('user_id', 'count'),
    conversions = ('converted', 'sum'),
    bounced = ('bounced', 'sum')
).reset_index()

# Calc conversion rate (CVR)
channel_summary['conversion_rate'] = (
    channel_summary['conversions'] / channel_summary['users']
)*100

# Calc bounce rate
channel_summary['bounce_rate'] = (
    channel_summary['bounced'] / channel_summary['users']
)*100

# Visualize CVR
fig = px.bar(
    channel_summary,
    x = 'channel',
    y = 'conversion_rate',
    color = 'variant',
    barmode = 'group',
    text='conversion_rate',
    title='<b>Conversion Rate (CVR) by Marketing Channel<b>'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')

fig.update_layout(
    yaxis_title='Conversion Rate (%)',
    xaxis_title='Marketing Channel',
    height=500
)

fig.show()

# Visualize bounce rate
fig = px.bar(
    channel_summary,
    x = 'channel',
    y = 'bounce_rate',
    color = 'variant',
    barmode = 'group',
    text='bounce_rate',
    title='<b>Bounce Rate by Marketing Channel<b>'
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')

fig.update_layout(
    yaxis_title='Bounce Rate (%)',
    xaxis_title='Marketing Channel',
    height=500
)

fig.show()

### Pages Viewed

> Does the redesigned checkout affect user engagement or browsing behavior?



In [33]:
fig = px.histogram(
    df,
    x='pages_viewed',
    color='converted',
    barmode='overlay',
    marginal='box',
    nbins=20,
    opacity=0.7,
    title='<b>User Engagement by Conversion Status<b>'
)

fig.update_layout(
    yaxis_title='User Engagement',
    xaxis_title='Pages Viewed',
    height=500
)

fig.show()

In [ ]:
# NEXT: Manually create a repo; manually add this notebook